# Capitulo 17 - Avaliacao e Benchmarking de Sistemas RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap17_benchmarking.ipynb)

Neste capitulo, abordamos tecnicas de avaliacao e benchmarking para sistemas RAG.
Cobrimos metricas de recuperacao, qualidade de geracao e avaliacao end-to-end.

---

## Atribuicao e Fonte Original

**Fonte:** [RAG-with-Python-Cookbook](https://github.com/PacktPublishing/RAG-with-Python-Cookbook) - Packt Publishing

**Notebooks fonte:** ch10_rag_evaluation/rag_evaluation_techniques.ipynb, ch07_retrieval/retrieval_techniques.ipynb

**Adaptado e comentado por Allan Eric Jepsen** para o livro *LLM On-Premise: Guia Pratico*.

Todos os creditos aos autores originais sao mantidos.

---

## Instalacao de Dependencias

Execute a celula abaixo para instalar todos os pacotes necessarios.

In [ ]:
# Instalar dependencias do capitulo
%pip install -q pandas pyarrow fsspec openai chromadb tiktoken \
    sentence-transformers python-dotenv psycopg2-binary requests numpy

---
## Parte 1: Tecnicas de Recuperacao

Antes de avaliar, precisamos entender as tecnicas de recuperacao disponiveis.
Exploramos busca semantica, hibrida, re-ranking e outras estrategias.

## Pre-requisitos

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [ ]:
!pip install psycopg2==2.9.10
!pip install requests==2.32.3
!pip install openai==1.82.1
!pip install python-dotenv==1.0.0

### Carregar arquivos de exemplo

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the colab secrets and load it to the environmental variables.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Verificar if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### 7.1 Optimizing Query Results using Metadata Filtering in PostgreSQL

In [ ]:
import psycopg2
import json

POSTGRES_READY = False
conn = None
cur = None

try:
    conn = psycopg2.connect(
        dbname="rag_cookbook",
        user="rag_cookbook_user",
        password="rag_cookbook_user_pw",
        host="localhost",
        port="5432",
    )
    cur = conn.cursor()

    # Try enabling pgvector; if unavailable, keep notebook runnable without Docker.
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")

    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS embeddings_table_with_metadata (
            id SERIAL PRIMARY KEY,
            text_chunk TEXT,
            embedding VECTOR(1536),
            metadata JSONB
        )
        """
    )
    conn.commit()
    POSTGRES_READY = True
    print("PostgreSQL with pgvector is ready.")
except Exception as exc:
    print(f"Skipping PostgreSQL vector setup: {exc}")
    if conn is not None:
        conn.rollback()

### 7.2 Loading Metadata Chunks for Filtering

In [ ]:
# Definir text chunks and metadata
text_chunks = [
    {
        "text": "Roger Federer has won 20 Grand Slam titles in tennis.",
        "topic": "tennis",
    },
    {
        "text": "The FIFA World Cup is the most prestigious football tournament.",
        "topic": "football",
    },
    {
        "text": "Serena Williams is one of the greatest tennis players of all time.",
        "topic": "tennis",
    },
    {
        "text": "Lionel Messi has won multiple Ballon d'Or awards in football.",
        "topic": "football",
    },
]

if POSTGRES_READY and conn is not None and cur is not None:
    # Inicializar OpenAI client
    client = OpenAI()

    # Insert text chunks with embeddings and metadata
    for chunk in text_chunks:
        response = client.embeddings.create(
            input=chunk["text"],
            model="text-embedding-3-small",
        )
        embedding = response.data[0].embedding

        metadata = {"topic": chunk["topic"]}
        cur.execute(
            "INSERT INTO embeddings_table_with_metadata (text_chunk, embedding, metadata) VALUES (%s, %s, %s)",
            (chunk["text"], embedding, json.dumps(metadata)),
        )

    conn.commit()
else:
    print("Skipping metadata insert because PostgreSQL/pgvector is not available.")

In [ ]:
hypothetical_documents if "hypothetical_documents" in globals() else []

### 7.3 Improving Search Results with Multi-Query Retrieval

In [ ]:
from openai import OpenAI
from pydantic import BaseModel
import os

client = OpenAI()

question = "What are the benefits of renewable energy?"

query_prompt = f"""You are an AI language model assistant. Your task is
to create three alternative versions of the provided user query to
enhance the retrieval of relevant documents from a vector database.
By offering diverse variations of the query, your goal is to help
mitigate the limitations of distance-based similarity search. Provide
these alternative queries, each on a new line.

Original query: {question}
"""

# send the query prompt to OpenAI
class QueryVariations(BaseModel):
    queries: list[str]

completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[
        {
            "role": "user",
            "content": query_prompt,
        },
    ],
    response_format=QueryVariations,
)

queries = completion.choices[0].message.parsed.queries
queries

### 7.4 Applying Metadata Filters at Query Time

In [ ]:
# Consultar the table with metadata filtering
query = "Who is the best player?"
topic_filter = "football"

if POSTGRES_READY and conn is not None and cur is not None:
    response = client.embeddings.create(
        input=query,
        model="text-embedding-3-small",
    )
    query_embedding = response.data[0].embedding

    cur.execute(
        """
        SELECT text_chunk, 1 - (embedding <=> %s::vector) AS similarity
        FROM embeddings_table_with_metadata
        WHERE metadata->>'topic' = %s
        ORDER BY similarity DESC
        LIMIT 5
        """,
        (query_embedding, topic_filter),
    )
    results = cur.fetchall()
else:
    results = []
    print("Skipping metadata filtering query because PostgreSQL/pgvector is not available.")

results

### 7.5 Enhancing Search Results by Extending the Original Query with Generated Pseudo-Documents (HyDE)

In [ ]:
from pydantic import BaseModel
from openai import OpenAI

user_query = "What is the revenue of Company X in 2024?"

client = OpenAI()

class HypotheticalDocuments(BaseModel):
    documents: list[str]

prompt = f"""
You are an AI assistant. Based on the user query below, generate
three hypothetical text chunks that contain relevant information to
answer the query.
"""

completion = client.beta.chat.completions.parse(
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": user_query},
    ],
    model="gpt-5-mini",
    response_format=HypotheticalDocuments,
)

hypothetical_documents = completion.choices[0].message.parsed.documents
hypothetical_documents

### 7.6 Increasing Search Efficiency by Designing an Auto-Merging Retriever (aka Parent Document Retriever)

In [ ]:
file_path = (
    "../datasets/text_files/"
    "random-text-about-5-different-stories-with-paragraphs.txt"
)

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

leaf_node_size = 250
parent_merge = 4
parent_node_size = parent_merge * leaf_node_size
text_chunks = []

for leaf_node_id in range(0, len(text) // leaf_node_size, 1):
    parent_node_id = leaf_node_id // parent_merge
    leaf_chunk_start = leaf_node_id * leaf_node_size
    parent_chunk_start = parent_node_id * parent_node_size

    text_chunks.append(
        {
            "leaf_node_id": leaf_node_id,
            "leaf_chunk": text[
                leaf_chunk_start : leaf_chunk_start + leaf_node_size
            ],
            "parent_node_id": parent_node_id,
            "parent_chunk": text[
                parent_chunk_start : parent_chunk_start + parent_node_size
            ],
        }
    )

if POSTGRES_READY and conn is not None and cur is not None:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS auto_merging_retriever_text_chunks (
            leaf_node_id INTEGER PRIMARY KEY,
            leaf_chunk TEXT,
            parent_node_id INTEGER,
            parent_chunk TEXT,
            leaf_chunk_embedding VECTOR(1536)
        )
        """
    )
    conn.commit()

    client = OpenAI()
    cur.execute("DELETE FROM auto_merging_retriever_text_chunks")

    for text_chunk in text_chunks:
        leaf_embedding = client.embeddings.create(
            input=[text_chunk["leaf_chunk"]],
            model="text-embedding-3-small",
        ).data[0].embedding

        cur.execute(
            """
            INSERT INTO auto_merging_retriever_text_chunks
            (leaf_node_id, leaf_chunk, parent_node_id, parent_chunk, leaf_chunk_embedding)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (leaf_node_id) DO UPDATE
            SET leaf_chunk = EXCLUDED.leaf_chunk,
                parent_node_id = EXCLUDED.parent_node_id,
                parent_chunk = EXCLUDED.parent_chunk,
                leaf_chunk_embedding = EXCLUDED.leaf_chunk_embedding
            """,
            (
                text_chunk["leaf_node_id"],
                text_chunk["leaf_chunk"],
                text_chunk["parent_node_id"],
                text_chunk["parent_chunk"],
                leaf_embedding,
            ),
        )
    conn.commit()

    # Exemplo retrieval over leaf nodes with parent merge
    query = "Summarize the main idea of one of the stories."
    query_embedding = client.embeddings.create(
        input=[query],
        model="text-embedding-3-small",
    ).data[0].embedding

    cur.execute(
        """
        SELECT leaf_node_id, parent_node_id, leaf_chunk
        FROM auto_merging_retriever_text_chunks
        ORDER BY leaf_chunk_embedding <=> %s::vector
        LIMIT 8
        """,
        (query_embedding,),
    )
    top_leaf_hits = cur.fetchall()

    parent_counts = {}
    for _, parent_node_id, _ in top_leaf_hits:
        parent_counts[parent_node_id] = parent_counts.get(parent_node_id, 0) + 1

    merged_parent_ids = [
        parent_node_id for parent_node_id, count in parent_counts.items() if count >= 2
    ]

    if merged_parent_ids:
        cur.execute(
            """
            SELECT DISTINCT parent_node_id, parent_chunk
            FROM auto_merging_retriever_text_chunks
            WHERE parent_node_id = ANY(%s)
            ORDER BY parent_node_id
            """,
            (merged_parent_ids,),
        )
        auto_merged_context = cur.fetchall()
    else:
        auto_merged_context = []

    auto_merged_context
else:
    print("Skipping auto-merging demo because PostgreSQL/pgvector is not available.")

### 7.7 Increasing Search Results by Designing a Sentence Window Retriever

In [ ]:
if POSTGRES_READY and conn is not None and cur is not None:
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS sentence_window_retriever_text_chunks (
            chunk_id SERIAL PRIMARY KEY,
            chunk TEXT,
            chunk_embedding VECTOR(1536)
        )
        """
    )
    conn.commit()

    # Demo data for sentence-window retrieval
    sentence_window_chunks = [
        "FC Bayern Munich is one of the most successful football clubs in Germany.",
        "The club was founded in 1900 and is based in Munich.",
        "Bayern has won many Bundesliga titles and UEFA trophies.",
        "Their squad often includes world-class international players.",
        "The team plays home matches at the Allianz Arena.",
    ]

    # Reset table for reproducible runs in this demo cell
    cur.execute("TRUNCATE sentence_window_retriever_text_chunks RESTART IDENTITY")
    client = OpenAI()

    for chunk in sentence_window_chunks:
        chunk_embedding = client.embeddings.create(
            input=[chunk],
            model="text-embedding-3-small",
        ).data[0].embedding
        cur.execute(
            """
            INSERT INTO sentence_window_retriever_text_chunks
            (chunk, chunk_embedding)
            VALUES (%s, %s)
            """,
            (chunk, chunk_embedding),
        )
    conn.commit()

    # Find the most relevant chunk and return a 3-chunk sentence window
    query = "Where does FC Bayern play?"
    query_embedding = client.embeddings.create(
        input=[query],
        model="text-embedding-3-small",
    ).data[0].embedding

    cur.execute(
        """
        SELECT chunk_id, chunk
        FROM sentence_window_retriever_text_chunks
        ORDER BY chunk_embedding <=> %s::vector
        LIMIT 1
        """,
        (query_embedding,),
    )
    top_hit = cur.fetchone()

    if top_hit is not None:
        center_chunk_id = top_hit[0]
        cur.execute(
            """
            SELECT chunk_id, chunk
            FROM sentence_window_retriever_text_chunks
            WHERE chunk_id BETWEEN %s AND %s
            ORDER BY chunk_id
            """,
            (center_chunk_id - 1, center_chunk_id + 1),
        )
        sentence_window = cur.fetchall()
    else:
        sentence_window = []

    sentence_window
else:
    print("Skipping sentence-window demo because PostgreSQL/pgvector is not available.")

### 7.8 Addressing Complex Requests by Designing a Query Routing System

In [ ]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

user_queries = [
    {
        "query": "Who is the all-time top scorer in the FIFA World Cup?",
        "selected_data_source": None,
    },
    {
        "query": "What are the four Grand Slam tennis tournaments?",
        "selected_data_source": None,
    },
    {
        "query": "Did Manchester United win their last game?",
        "selected_data_source": None,
    },
]

prompt = f"""
You are an expert at routing a user question to the appropriate
data source. Given a user question choose which of the data sources
in list_of_data_sources is the best to answer the question.
"""

from typing import Literal
from pydantic import Field

class RouterDecision(BaseModel):
    data_source: Literal[
        "general_football_knowledge",
        "general_tennis_knowledge",
        "latest_football_results_sql",
    ] = Field(
        ...,
        description="The best data source to answer the question."
    )

for user_query in user_queries:
    completion = client.beta.chat.completions.parse(
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": user_query["query"]},
        ],
        model="gpt-5-mini",
        response_format=RouterDecision,
    )
    user_query["selected_data_source"] = completion.choices[
        0
    ].message.parsed.data_source

completion

### 7.9 Improving Search Accuracy with Reranking Methods

In [ ]:
import textwrap
from pydantic import BaseModel
from openai import OpenAI

text_chunks = {
    1: (
        "Tesla's Supercharger network and tech lead face rising "
        "competition from BYD and established automakers."
    ),
    2: (
        "Tesla's production grows, but price competition threatens "
        "its market share."
    ),
    3: (
        "The automotive industry is shifting to EVs due to climate "
        "change and regulations."
    ),
    4: "Semiconductor shortages are disrupting automotive supply chains.",
    5: (
        "Consumer demand for autonomous driving and advanced tech "
        "impacts EV competition."
    ),
}

prompt = textwrap.dedent(
    f"""
    Query: Will Tesla remain the market leader in electric vehicles?

    Documents:
        1. {text_chunks[1]}
        2. {text_chunks[2]}
        3. {text_chunks[3]}
        4. {text_chunks[4]}
        5. {text_chunks[5]}

    Instructions:
    Please assess the relevance of each document to the query and
    provide a relevance score from 1 to 5, where 5 is the most relevant.
    """
)

class RelevanceScore(BaseModel):
    document_id: int
    relevance_score: int

class RelevanceScores(BaseModel):
    relevance_scores: list[RelevanceScore]

client = OpenAI()
completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": prompt}],
    response_format=RelevanceScores,
)

reranking_results = completion.choices[0].message.parsed.relevance_scores
sorted(
    reranking_results,
    key=lambda x: x.relevance_score,
    reverse=True,
    )

### 7.10 Decomposing Complex Queries into Multiple Sub-Queries

In [ ]:
from pydantic import BaseModel 
from typing import Optional 
from openai import OpenAI

class Question(BaseModel):
    question: str
    answer: Optional[str] = None

class Questions(BaseModel): 
    questions: list[Question]

splitter_prompt = """
You are a helpful assistant for a RAG chatbot.

Your job is to break down complex questions into simpler ones that 
are easy to answer. When the answers to these simpler questions are 
combined, they should fully answer the original question. If the 
question is already simple, leave it as it is. Handle one question 
at a time.

Example:
    1. Query: Did Microsoft or Google make more money last year?

Decomposed Questions:
    1. How much profit did Microsoft make last year?
    2. How much profit did Google make last year? 
"""

client = OpenAI()

query = (
    "What are the benefits of renewable energy compared to "
    "fossil fuels?"
)

completion = client.beta.chat.completions.parse(
    model="gpt-5-mini",
        messages=[
        {"role": "system", "content": splitter_prompt}, 
        {"role": "user", "content": query},
    ], 
    response_format=Questions,
)

decomposed_questions = completion.choices[0].message.parsed.questions
decomposed_questions


---
## Parte 2: Avaliacao de Sistemas RAG

Avaliamos o sistema RAG com metricas objetivas: precisao, recall, F1,
faithfulness (fidelidade), relevancia e coerencia da resposta.

## Pre-requisitos

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [ ]:
!pip install pandas
!pip install pyarrow
!pip install fsspec
!pip install huggingface_hub
!pip install pydantic
!pip install openai

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the colab secrets and load it to the environmental variables.

In [ ]:
import os
from dotenv import load_dotenv

# Support both Google Colab secrets and local .env files.
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        api_key = None

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or Colab Secrets")

os.environ["OPENAI_API_KEY"] = api_key

### 1. Creating Synthetic Data for Automated Testing

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "hf://datasets/rag-datasets/rag-mini-wikipedia/data/"
    "passages.parquet/part.0.parquet"
)

docs_list = df.passage.to_list()

In [ ]:
print(f"Number of documents: {len(docs_list)}")
print(f"List of docs:{docs_list}")

**Busca semantica:** Buscamos os documentos mais relevantes por similaridade.

In [ ]:
from pydantic import BaseModel

QA_generation_prompt = """
    Your task is to generate a question and its corresponding answer based on the
    provided context.

    The question should be:
    - Direct and focused, seeking a specific factual piece of information present
        in the context.
    - Formulated as a natural query a user might input into a search engine.
    - Independent of the context, meaning it should not contain phrases like
        "according to the passage" or "in the context".

    The answer should be:
    - A concise and accurate response directly extracted or inferred from the context.

    Present your output in the following format:

    Output:
    Question: (Your concise, fact-based question)
    Answer: (The direct answer to the question)

    Context: {context}\n
    Output:"""


**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:

from openai import OpenAI
from pydantic import Field
import random

# Inicializar the OpenAI client with your API key
client = OpenAI()

class QuestionAnswerPairs(BaseModel):
    question: str = Field(description="A factoid question about the context.")
    answer: str = Field(description="The answer to the factoid question.")

question_answer_pairs = []

random_text_chunks = random.sample(docs_list, 5)

for text_chunk in random_text_chunks:
    # generate hypothetical questions using the GPT-4 model
    completion = client.chat.completions.parse(
        messages=[
            {
                "role": "user",
                "content": QA_generation_prompt.format(context=text_chunk),
            }
        ],
        model="gpt-5.2",
        response_format=QuestionAnswerPairs,
    )

    question_answer_pair = completion.choices[0].message.parsed.model_dump()
    question_answer_pair["context"] = text_chunk
    question_answer_pairs.append(question_answer_pair)

In [ ]:
question_answer_pairs

### 2. Evaluating the Retriever step by calculating the Context Precision@k

In [ ]:
question = "What is the capital of France?"
answer = "The capital of France is Paris."
retrieved_contexts = ["""France, a country in Western Europe, is known for its capital city,
    Paris, which is a major European city and a global center for art, fashion,
    and culture.""",
    """Paris is the capital city of France and is renowned for its historical landmarks
    such as the Eiffel Tower and the Louvre Museum.""",
    """The Amazon rainforest is the world's largest tropical rainforest, spanning
    across nine South American countries and supporting immense biodiversity."""
]

**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:
from textwrap import dedent
from pydantic import BaseModel, Field
from openai import OpenAI

def context_relevance_verification(question, answer, retrieved_contexts):
    # Pydantic model for verification
    class Verification(BaseModel):
        reason: str = Field(..., description="Reason for verification")
        verdict: int = Field(..., description="Binary (0/1) verdict of verification")

    # Definir the prompt for context relevance verification
    context_relevance_prompt = dedent('''
        Given a question, an answer, and a context, verify if the context
        was instrumental in deriving the given answer. Provide a detailed reason
        for your assessment and a binary verdict: "1" if the context is useful
        and "0" if it is not.

        Input:
        Question: "{question}",
        Answer: "{answer}",
        Context: "{context}"
        ''')

    list_of_verifications = []

    # Iterate through each question-answer pair and its context
    for retrieved_context in retrieved_contexts:
        prompt = context_relevance_prompt.format(
                question=question,
                answer=answer,
                context=retrieved_context)

        client = OpenAI()
        completion = client.chat.completions.parse(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="gpt-5.2",
            response_format=Verification,
        )

        verification = completion.choices[0].message.parsed.model_dump()
        verification["question"] = question
        verification["answer"] = answer
        verification["retrieved_context"] = retrieved_context

        list_of_verifications.append(verification)
    return list_of_verifications

In [ ]:
list_of_verifications = context_relevance_verification(
                                    question,
                                    answer,
                                    retrieved_contexts
                                )
list_of_verifications

In [ ]:
import pandas as pd

def calculate_context_relevance_score(list_of_verifications):
    verdict_list = pd.DataFrame(list_of_verifications)["verdict"].to_list()

    denominator = sum(verdict_list) + 1e-10
    numerator = sum(
        [
            (sum(verdict_list[: i + 1]) / (i + 1)) * verdict_list[i]
            for i in range(len(verdict_list))
        ]
    )
    score = numerator / denominator
    return score

In [ ]:
score = calculate_context_relevance_score(list_of_verifications)
score

### 3. Evaluating RAG systems using LLMs as a judge and the Faithfulness Metrics

    """
    Decomposes the sentences in an answer into simpler, unambiguous statements.

    Args:
        question: The original question.
        answer: The answer to be decomposed.
        client: An initialized OpenAI client.

    Returns:
        A list of decomposed statements.
    """

In [ ]:
from textwrap import dedent
from pydantic import BaseModel, Field
from openai import OpenAI

def decompose_answer(question, answer):
    # Inicializar the OpenAI client
    client = OpenAI()

    statement_prompt = """
    Given a question and an answer, analyze the complexity of each sentence
    in the answer. Break down each sentence into one or more fully
    understandable statements. Ensure that no pronouns or ambiguous references
    are used in any statement. Output the decomposed statements as a list of strings.

    Question: {question}
    Answer: {answer}
    """

    prompt = statement_prompt.format(question=question, answer=answer)

    class Statements(BaseModel):
        """Structured response for statement extraction."""

        statements: list[str] = Field(description="List of decomposed statements.")

    completion = client.chat.completions.parse(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="gpt-5.2",
        response_format=Statements,
    )

    statements = completion.choices[0].message.parsed.statements
    return statements

In [ ]:
question = """Describe the geographical extent and ecological significance of the
Amazon rainforest."""

answer = dedent("""
The Amazon rainforest, the world's largest tropical rainforest, covers significant
territory across nine South American countries. This vast area is renowned for
its immense biodiversity, supporting millions of different plant and animal species.
While it plays a crucial role in the global climate by absorbing substantial amounts
of carbon dioxide, its impact on the overall oxygen production of the Earth is often
overstated.
""")

In [ ]:
statements = decompose_answer(question, answer)

print("Decomposed Statements:")
for idx, stmt in enumerate(statements, 1):
    print(f"{idx}. {stmt}")

**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:
from pydantic import BaseModel, Field
from textwrap import dedent
from openai import OpenAI

def judge_faithfulness(statement: str, context: str) -> list[dict]:

    # Inicializar the OpenAI client
    client = OpenAI()

    faithfulness_judge_prompt = dedent(
        """
        Your task is to judge the faithfulness of a statement based on the
        given context. You must return a verdict as 1 if the statement can
        be directly inferred from the context, or 0 if the statement cannot
        be directly inferred. Explain your reasoning.

        Context:
        {context}

        Statement:
        {statement}

        Answer:::
        Reason: (Explain your reasoning)
        Verdict: (1 or 0)
        """
    )

    class StatementFaithfulness(BaseModel):
        """Structured response for statement extraction."""

        statements: str = Field(description="Decomposed statement.")
        reason: str = Field(description="Reasoning for the faithfulness judgement.")
        verdict: int = Field(
                        description="1 if the statement is faithful, 0 otherwise."
                        )

    prompt = faithfulness_judge_prompt.format(statement=statement, context=context)

    completion = client.chat.completions.parse(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="gpt-5.2",
        response_format=StatementFaithfulness,
    )

    statement_faithfulness = completion.choices[0].message.parsed.model_dump()
    return statement_faithfulness

In [ ]:
statements_faithfulness = []

context = dedent(
    """
    The Amazon rainforest is the largest tropical rainforest in the world,
    covering parts of nine South American countries. It is known for its
    incredible biodiversity, housing millions of species of plants, insects,
    birds, and other animals. The forest plays a crucial role in regulating
    the Earth's climate by absorbing vast amounts of carbon dioxide.
    """
)

for statement in statements:
    statement_faithfulness = judge_faithfulness(
        statement=statement, context=context
    )
    # Append the statement faithfulness to the list
    statements_faithfulness.append(statement_faithfulness)

In [ ]:
print("Number of statements:", len(statements_faithfulness))
statements_faithfulness

In [ ]:
import pandas as pd

statements_faith_df = pd.DataFrame(statements_faithfulness)

number_of_claims = len(statements_faith_df["verdict"])
number_of_claims_in_context = statements_faith_df["verdict"].sum()
faithfulness_percentage = (
    number_of_claims_in_context / number_of_claims
) * 100

In [ ]:
faithfulness_percentage

### 4. RAG evaluation using Response Relevancy

In [ ]:
user_input="What is the capital of France?"
response="The capital of France is Paris."

In [ ]:
from textwrap import dedent

response_relevance_prompt = dedent("""
    Generate three relevant question for the following answer
    and indicate if the answer is noncommittal.

    Output:
    Question: [Generated Question]
    Noncommittal (1=Yes, 0=No): [0 or 1]

    An answer is considered noncommittal if it is evasive, vague, or ambiguous
    (e.g., "I don't know," "Maybe," "It depends").

    Answer: {response}
    """)

**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:
from pydantic import BaseModel, Field

prompt = response_relevance_prompt.format(response=response)

class GeneratedQuestion(BaseModel):
    question: str = Field(description="Generated question.")
    noncommittal: int = Field(
        description="1 if the answer is noncommittal, 0 otherwise."
    )

class GeneratedQuestions(BaseModel):
    questions: list[GeneratedQuestion] = Field(
        description="List of generated questions and their noncommittal status."
    )

client = OpenAI()

completion = client.chat.completions.parse(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="gpt-5.2",
    response_format=GeneratedQuestions,
)

generated_questions = completion.choices[0].message.parsed.model_dump()["questions"]

In [ ]:
generated_questions

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
from openai import OpenAI

client = OpenAI()

def create_embeddings(text_chunk, client):
    embed_model = "text-embedding-3-small"
    embedding = (
        client.embeddings.create(input=[text_chunk], model=embed_model)
        .data[0]
        .embedding
    )
    return embedding

In [ ]:
user_input

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
user_question_embedding = create_embeddings(user_input, client)
cosine_sims = []

**Geracao de embeddings:** Criamos representacoes vetoriais do texto.

In [ ]:
import numpy as np
import pandas as pd

user_question_embedding = create_embeddings(user_input, client)
cosine_sims = []

for generated_question in generated_questions:
    generated_question["embedding"] = create_embeddings(
        generated_question["question"], client
    )

    generated_question["cosine_sim"] = np.dot(
        user_question_embedding, generated_question["embedding"]
    ) / (
        np.linalg.norm(user_question_embedding)
        * np.linalg.norm(generated_question["embedding"])
    )

In [ ]:
generated_question

In [ ]:
# calculate the mean cosine similarity over all generated questions
cosine_sim_mean = pd.DataFrame(generated_questions)["cosine_sim"].mean()

# check if any of the generated questions are non-committal
committal = any(pd.DataFrame(generated_questions)["noncommittal"])

# if any of the generated questions are non-committal, set the score to 0
response_relevance_score = cosine_sim_mean * int(not committal)

In [ ]:
response_relevance_score